# Matcher Validation

Check connection coverage, fuzzy-match quality, and warmth distribution.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import select
from sqlalchemy.orm import Session
from src.db import get_engine, Posting
from src.match import _load_connections, _normalize, CONNECTIONS_PATH
from pathlib import Path

engine = get_engine(Path('../data/jobs.db'))
connections_path = Path('../data/Connections.csv')
pd.set_option('display.max_colwidth', 80)

## Connection coverage

In [ ]:
with Session(engine) as s:
    matched = s.scalars(select(Posting).where(Posting.status.in_(['matched','generated','reviewed','applied']))).all()

df = pd.DataFrame([
    {
        'id': p.id,
        'company': p.company,
        'title': p.title,
        'score': p.relevance_score,
        'connections': json.loads(p.connections_json or '[]'),
    }
    for p in matched
])
df['n_conns'] = df['connections'].apply(len)
df['has_conn'] = df['n_conns'] > 0

print(f'Total matched postings:  {len(df)}')
print(f'With ≥1 connection:      {df["has_conn"].sum()} ({df["has_conn"].mean():.0%})')
print(f'Cold (no connection):    {(~df["has_conn"]).sum()}')

fig, ax = plt.subplots(figsize=(8, 3))
counts = df['n_conns'].value_counts().sort_index()
ax.bar(counts.index, counts.values, color='#4c9be8')
ax.set_xlabel('Connections found')
ax.set_ylabel('Postings')
ax.set_title('Connections per matched posting')
plt.tight_layout()
plt.show()

## Postings with connections — ranked by score

In [ ]:
warm = df[df['has_conn']].sort_values('score', ascending=False).copy()
warm['top_connection'] = warm['connections'].apply(
    lambda cs: f"{cs[0]['name']} ({cs[0]['position']})" if cs else ''
)
warm[['score', 'company', 'title', 'n_conns', 'top_connection']].head(20)

## Spot-check fuzzy matching

Verify the company normalizer is working correctly for specific names.

In [ ]:
test_pairs = [
    ('Apple Inc.', 'Apple'),
    ('Meta Platforms', 'Meta'),
    ('Google LLC', 'Google'),
    ('Checkout.com', 'Checkout.com'),
    ('DeepMind Technologies', 'DeepMind'),
]
print('Normalization check:')
for raw, expected in test_pairs:
    got = _normalize(raw)
    ok = '✓' if got == expected.lower() else '✗'
    print(f'  {ok}  "{raw}" → "{got}" (expected "{expected.lower()}")')

## Live fuzzy match — query your connections for a company

In [ ]:
from rapidfuzz import fuzz, process

QUERY_COMPANY = 'Apple'  # ← change this
THRESHOLD = 80

if connections_path.exists():
    connections = _load_connections(connections_path)
    norms = [_normalize(c['Company']) for c in connections]
    results = process.extract(_normalize(QUERY_COMPANY), norms, scorer=fuzz.ratio, limit=None)
    hits = [(connections[idx], score) for _, score, idx in results if score >= THRESHOLD]
    hits.sort(key=lambda x: x[1], reverse=True)
    print(f'Connections matching "{QUERY_COMPANY}" (threshold={THRESHOLD}):')
    for c, score in hits:
        print(f'  [{score:3.0f}] {c["First Name"]} {c["Last Name"]:20s} — {c.get("Position","")} @ {c["Company"]}')
else:
    print(f'Connections CSV not found at {connections_path}')

## Inspect all connections for a specific posting

In [ ]:
# Change posting_id to any id from the table above
posting_id = int(warm.iloc[0]['id']) if len(warm) else None

if posting_id:
    with Session(engine) as s:
        p = s.get(Posting, posting_id)
    conns = json.loads(p.connections_json or '[]')
    print(f'{p.company} — {p.title}')
    print(f'URL: {p.url}\n')
    if conns:
        for c in conns:
            print(f"  {c['name']:30s}  {c['position']:40s}  {c['linkedin_url']}")
    else:
        print('  (no connections found)')
else:
    print('No warm postings yet.')